In [ ]:
#====================================================================================================#
#                                                                                                    #
#                                                        ██╗   ██╗   ████████╗ █████╗ ██████╗        #
#      AEC3 - BAIN                                       ██║   ██║   ╚══██╔══╝██╔══██╗██╔══██╗       #
#                                                        ██║   ██║█████╗██║   ███████║██║  ██║       #
#      created:        22/05/2026  -  01:00:51           ██║   ██║╚════╝██║   ██╔══██║██║  ██║       #
#      last change:    31/05/2026  -  22:14:43           ╚██████╔╝      ██║   ██║  ██║██████╔╝       #
#                                                         ╚═════╝       ╚═╝   ╚═╝  ╚═╝╚═════╝        #
#                                                                                                    #
#      Ismael Hernandez Clemente                         ismael.hernandez@live.u-tad.com             #
#                                                                                                    #
#      Github:                                           https://github.com/ismaelucky342            #
#                                                                                                    #
#====================================================================================================# 

# AEC3 – Análisis de Redes e Integración de LLMs

Esta entrega amplía la clase `DataExtractor` de la AEC2 con dos bloques nuevos:

1. **Análisis de redes sociales con NetworkX**: grafo de menciones, métricas de centralidad y detección de comunidades (Louvain).
2. **Integración de LLM local via Ollama**: los insights de la red se convierten en un prompt interpretativo que se pasa a `gemma3` corriendo en local.

Todo sigue centralizado en la misma clase `DataExtractor`.

## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo la primera vez
# !pip install pandas requests textblob nltk gensim spacy spacytextblob wordcloud matplotlib seaborn networkx python-louvain streamlit python-dotenv
# !python -m spacy download en_core_web_sm
# !python -m textblob.download_corpora

# Ollama: instalar desde https://ollama.com y ejecutar:
# ollama serve
# ollama pull gemma3

## 1. Imports y carga de la clase

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from dotenv import load_dotenv

# Cargar variables de entorno (.env con RAPIDAPI_KEY)
load_dotenv()

# Importar la clase principal
from data_extractor import DataExtractor

print("✅ Imports OK")

## 2. Carga de datos

Dos opciones: API en tiempo real o dataset local de Kaggle.
Para el análisis de redes se recomienda el dataset de Kaggle (más tweets = grafo más rico).

In [ ]:
# ── Opción A: desde la API de Twitter (RapidAPI) ──────────────────────────────
# ext = DataExtractor()
# ext.load_data_api(query="#bitcoin", max_results=200, output_file="tweets_api.csv")

# ── Opción B: desde CSV local (recomendado para análisis de redes) ────────────
ext = DataExtractor(source="Bitcoin_tweets.csv")
ext.load_data()

print(f"\nColumnas disponibles: {list(ext.data.columns)}")
ext.data.head(3)

## 3. Análisis de hashtags (AEC1 – heredado)

In [ ]:
analytics = ext.analytics_hashtags_extended()
print("\nTop 10 hashtags:")
analytics['overall'].head(10)

In [ ]:
ext.generate_hashtag_wordcloud(analytics['overall'], max_words=80)

## 4. Análisis de sentimiento (AEC2 – heredado)

In [ ]:
ext.analyze_sentiment(method='textblob')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(ext.data['sentiment_polarity'].dropna(), bins=40,
             color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.2, label='Neutral')
axes[0].set_xlabel('Polaridad')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de polaridad')
axes[0].legend()

axes[1].hist(ext.data['sentiment_subjectivity'].dropna(), bins=40,
             color='darkorange', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Subjetividad')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de subjetividad')

plt.suptitle('Análisis de sentimiento – Bitcoin Tweets', fontsize=13)
plt.tight_layout()
plt.show()

pol_mean = ext.data['sentiment_polarity'].mean()
print(f"Polaridad media: {pol_mean:.4f} {'(positivo)' if pol_mean > 0 else '(negativo)'}")

## 5. Construcción del grafo de interacciones (AEC3 – NUEVO)

### ¿Por qué un grafo de menciones?

Las menciones `@usuario` en Twitter son el mecanismo de interacción directa entre cuentas. Modelar estas interacciones como un grafo dirigido (A menciona a B = arista A→B) nos permite:

- Identificar quiénes son los nodos más influyentes (centralidad).
- Detectar grupos de usuarios que interactúan preferentemente entre sí (comunidades).
- Entender la estructura de difusión de información en la red.

In [ ]:
# Construir el grafo de menciones
G = ext.build_interaction_graph()

print(f"\nEstadísticas básicas del grafo:")
print(f"  Nodos    : {G.number_of_nodes():,}")
print(f"  Aristas  : {G.number_of_edges():,}")
print(f"  Densidad : {nx.density(G):.6f}")
print(f"  Es DAG   : {nx.is_directed_acyclic_graph(G)}")

## 6. Cálculo de métricas y detección de comunidades (AEC3 – NUEVO)

### Métricas de centralidad

| Métrica | Definición | Uso |
|---|---|---|
| **In-degree centrality** | Proporción de nodos que apuntan a este nodo | ¿Quién es más mencionado? |
| **Betweenness centrality** | Frecuencia con la que el nodo aparece en los caminos más cortos entre pares de nodos | ¿Quién actúa como puente? |
| **PageRank** | Importancia ponderada por la importancia de los que te señalan | ¿Quién tiene influencia real? |

### Detección de comunidades

El algoritmo de **Louvain** maximiza la modularidad de la partición: busca grupos donde las aristas internas son más densas que lo esperado por azar. Es el estándar de facto para grafos de redes sociales grandes.

In [ ]:
# Calcular métricas y generar visualización
metrics = ext.analyze_network(G)

In [ ]:
# Visualizar top usuarios por métricas en una tabla comparativa
pagerank  = metrics['pagerank']
in_deg    = metrics['in_degree_centrality']
betw      = metrics['betweenness']

# Top 10 por PageRank
top10_nodes = sorted(pagerank, key=pagerank.get, reverse=True)[:10]

df_top = pd.DataFrame({
    'Usuario'      : [f"@{n}" for n in top10_nodes],
    'PageRank'     : [round(pagerank[n], 6) for n in top10_nodes],
    'In-degree'    : [round(in_deg[n],   6) for n in top10_nodes],
    'Betweenness'  : [round(betw.get(n, 0), 6) for n in top10_nodes],
    'Tweets'       : [G.nodes[n].get('tweet_count', 0) for n in top10_nodes],
})

print("Top 10 nodos por PageRank:")
df_top

In [ ]:
# Distribución del número de comunidades
from collections import Counter

comm_sizes = Counter(metrics['communities'].values())
sizes_sorted = sorted(comm_sizes.values(), reverse=True)[:20]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(sizes_sorted)), sizes_sorted, color='steelblue', edgecolor='white')
ax.set_xlabel('Comunidad (top 20 por tamaño)')
ax.set_ylabel('Número de nodos')
ax.set_title(f'Tamaño de comunidades detectadas (total: {metrics["n_communities"]})')
plt.tight_layout()
plt.show()

## 7. Generación del prompt desde la red (AEC3 – NUEVO)

Los insights cuantitativos del grafo se condensan en un prompt en lenguaje natural.
El objetivo es que el LLM aporte una interpretación contextual que vaya más allá de los números.

In [ ]:
prompt = ext.generate_prompt_from_network(G, metrics)
print(prompt)

## 8. Chat con LLM local – Ollama gemma3 (AEC3 – NUEVO)

### Requisitos previos

```bash
# Terminal 1: iniciar Ollama
ollama serve

# Terminal 2: descargar el modelo
ollama pull gemma3
```

### ¿Por qué Ollama en lugar de Hugging Face directamente?

Hugging Face permite más control (fine-tuning, acceso a pesos, etc.) pero requiere:
- Gestionar manualmente el tokenizador, el device map y la cuantización.
- Cargar el modelo entero en Python (pesado en RAM/VRAM).
- Gestionar la autenticación con tokens de HF.

Ollama encapsula todo eso: descarga el modelo cuantizado, lo ejecuta como un servidor local y expone una API REST limpia. El resultado es el mismo modelo pero con una integración mucho más simple y ligera para este tipo de pipeline.

### Justificación de 'steps' y 'training loss'

Si en el futuro se hiciera fine-tuning del modelo sobre tweets de Bitcoin:
- **steps**: número de actualizaciones de pesos durante el entrenamiento. Más steps = más adaptación al dominio, pero riesgo de sobreajuste.
- **training loss**: medida del error del modelo en los datos de entrenamiento. Debe descender de forma estable; si se estanca o sube, hay un problema de learning rate o datos.

In [ ]:
# ── Opción A: enviar el prompt de la red y entrar en modo chat interactivo ──
# NOTA: esta celda abre un bucle interactivo en el terminal/kernel.
# En Jupyter puede que prefieras usar el dashboard para el chat.

# last_response = ext.chat_local_llm(
#     prompt=prompt,
#     model="gemma3",
#     ollama_url="http://localhost:11434",
#     stream=True
# )

# ── Opción B: petición única sin modo interactivo (útil en notebooks) ────────
import requests, json

def ask_ollama_once(prompt_text: str, model: str = "gemma3",
                    url: str = "http://localhost:11434") -> str:
    """Hace una única petición a Ollama y devuelve la respuesta completa."""
    payload = {
        "model":    model,
        "messages": [{"role": "user", "content": prompt_text}],
        "stream":   False,
    }
    resp = requests.post(f"{url}/api/chat", json=payload, timeout=300)
    resp.raise_for_status()
    return resp.json().get('message', {}).get('content', '')

# Descomentar para ejecutar:
# response = ask_ollama_once(prompt)
# print(response)

print("Función ask_ollama_once definida. Descomenta las últimas líneas para ejecutarla.")
print("Asegúrate de que 'ollama serve' está en ejecución antes de llamarla.")

## 9. Resumen extractivo (AEC2 – heredado)

In [ ]:
summary = ext.parse_and_summarize(summary_ratio=0.02, max_length=300)
print("RESUMEN EXTRACTIVO DEL CORPUS\n" + "=" * 40)
print(summary)

## 10. Exportar todos los resultados

In [ ]:
# Exportar todo el pipeline
topics = None
try:
    topics = ext.model_topics(num_topics=5, passes=10)
    print("Tópicos LDA:")
    for i, t in enumerate(topics, 1):
        print(f"  Tópico {i}: {', '.join(t)}")
except ImportError as e:
    print(f"LDA no disponible: {e}")

ext.export_results(output_dir="output", topics=topics)

## 11. Dashboard Streamlit

Para lanzar el dashboard interactivo con las 6 tabs (incluyendo la red y el chat LLM):

```bash
streamlit run dashboard.py
```

El dashboard permite:
- Subir cualquier CSV con tweets.
- Explorar hashtags, wordcloud, evolución temporal y detección de bots.
- **NUEVO**: construir el grafo de menciones, ver métricas de centralidad y comunidades.
- **NUEVO**: generar el prompt desde la red y chatear con gemma3 directamente desde el navegador.